# SmartChef-AI: Pure NumPy Recipe Recommender Model Training & Serialization

This notebook handles the offline machine learning workflow without loading native C++ extensions like scikit-learn (to bypass Windows Application Control DLL blocks):
1. Loading recipe and ingredient datasets.
2. Cleaning and building a binary ingredient feature matrix.
3. Implementing Cosine Similarity matching using pure NumPy vector operations.
4. Evaluating model recommendations on sample query inputs.
5. Serializing and saving the feature matrix and metadata to disk for use in the main Streamlit application.

In [2]:
import os
import pickle
import pandas as pd
import numpy as np

In [3]:
# Load the datasets
master_df = pd.read_csv('../datasets/recipes_master.csv')
ing_df = pd.read_csv('../datasets/Recipe_ingrediect.csv')

print(f"Loaded {len(master_df)} recipes and {len(ing_df)} rows in ingredient mapping.")

Loaded 9997 recipes and 9997 rows in ingredient mapping.


In [4]:
# Drop duplicates in ingredients to ensure a clean 1-to-1 recipe mapping
ing_df_clean = ing_df.drop_duplicates(subset=['recipe_name']).copy()
ingredient_cols = [col for col in ing_df.columns if col != 'recipe_name']

# Build feature matrix X
X = ing_df_clean[ingredient_cols].values
total_ingredients = X.sum(axis=1)

print(f"Feature matrix shape: {X.shape} (Recipes: {X.shape[0]}, Ingredients: {X.shape[1]})")

Feature matrix shape: (3085, 58) (Recipes: 3085, Ingredients: 58)


In [5]:
# Test / Evaluation function using pure NumPy Vector Math
def test_query(user_ingredients, top_n=5):
    # Vectorize input
    query_vector = np.zeros(len(ingredient_cols))
    for user_ing in user_ingredients:
        for idx, col in enumerate(ingredient_cols):
            col_lower = col.lower()
            if user_ing == col_lower or user_ing in col_lower or col_lower in user_ing:
                query_vector[idx] = 1
                
    if np.sum(query_vector) == 0:
        print("No matching ingredients found in database.")
        return
        
    # Calculate match counts: dot product between binary matrix X and query vector
    match_counts = np.dot(X, query_vector)
    
    # Calculate Cosine Similarity
    norm_q = np.sqrt(np.sum(query_vector))
    norm_X = np.sqrt(total_ingredients)
    
    # Avoid division by zero
    denom = norm_q * np.where(norm_X > 0, norm_X, 1.0)
    match_scores = match_counts / denom
    
    # Find indices of top N matches sorted in descending order of score
    sorted_indices = np.argsort(match_scores)[::-1]
    
    # Print results
    print(f"\nRecommendations for: {', '.join(user_ingredients)}")
    for i in range(top_n):
        idx = sorted_indices[i]
        recipe_name = ing_df_clean.iloc[idx]['recipe_name']
        print(f"  - {recipe_name}: Match = {match_scores[idx]*100:.1f}%, Count = {int(match_counts[idx])}/{int(total_ingredients[idx])}")

# Run a test query
test_query(['onion', 'garlic', 'chicken'])


Recommendations for: onion, garlic, chicken
  - Pakistani Beverages Special 365: Match = 86.6%, Count = 3/4
  - Indian Appetizers Special 541: Match = 86.6%, Count = 3/4
  - Indian Main Course Special 95: Match = 77.5%, Count = 3/5
  - Indian Main Course Special 239: Match = 77.5%, Count = 3/5
  - Indian Main Course Special 268: Match = 77.5%, Count = 3/5


In [6]:
# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Package model features and metadata (no sklearn objects!)
model_data = {
    'ingredient_cols': ingredient_cols,
    'recipe_names': ing_df_clean['recipe_name'].values,
    'total_ingredients': total_ingredients,
    'X': X
}

# Serialize and save model
model_path = '../models/recommender_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(model_data, f)
    
print(f"Model saved successfully to {model_path}!")

Model saved successfully to ../models/recommender_model.pkl!
